In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = "/content/drive/MyDrive/NLP_dataset/61262-0.txt"

with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))
print(text[:1000])

Total characters: 296991
*** START OF THE PROJECT GUTENBERG EBOOK 61262 ***

  POIROT INVESTIGATES




  BY THE SAME AUTHOR

  THE MYSTERIOUS AFFAIR AT STYLES

  THE SECRET ADVERSARY

  THE MURDER ON THE LINKS

  THE BODLEY HEAD




  POIROT INVESTIGATES

  BY AGATHA CHRISTIE




  LONDON

  JOHN LANE THE BODLEY HEAD LIMITED




  First published in Great Britain by
  John Lane Company, The Bodley Head Limited, 1924

  Copyright © 1924 Agatha Christie Limited




  CONTENTS


  I The Adventure of “The Western Star”

  II The Tragedy at Marsdon Manor

  III The Adventure of the Cheap Flat

  IV The Mystery of Hunter’s Lodge

  V The Million Dollar Bond Robbery

  VI The Adventure of the Egyptian Tomb

  VII Jewel Robbery at the _Grand Metropolitan_

  VIII The Kidnapped Prime Minister

  IX The Disappearance of Mr. Davenheim

  X The Adventure of the Italian Nobleman

  XI The Case of the Missing Will




  POIROT INVESTIGATES




  POIROT INVESTIGATES


  I


  The Adventure of “The We

In [ ]:
import re

# Remove Gutenberg header
text = re.split(r"\*\*\* START OF.*?\*\*\*", text, maxsplit=1)[-1]

# Remove Gutenberg footer
text = re.split(r"\*\*\* END OF.*?\*\*\*", text, maxsplit=1)[0]

# Remove everything before "contents"
contents_index = text.lower().find("contents")
text = text[contents_index:]

# Remove everything after "the end"
end_index = text.lower().rfind("the end")
text = text[:end_index]

print(text[:1000])
print(text[-1000:])

CONTENTS


  I The Adventure of “The Western Star”

  II The Tragedy at Marsdon Manor

  III The Adventure of the Cheap Flat

  IV The Mystery of Hunter’s Lodge

  V The Million Dollar Bond Robbery

  VI The Adventure of the Egyptian Tomb

  VII Jewel Robbery at the _Grand Metropolitan_

  VIII The Kidnapped Prime Minister

  IX The Disappearance of Mr. Davenheim

  X The Adventure of the Italian Nobleman

  XI The Case of the Missing Will




  POIROT INVESTIGATES




  POIROT INVESTIGATES


  I


  The Adventure of “The Western Star”

I was standing at the window of Poirot’s rooms looking out idly on
the street below.

“That’s queer,” I ejaculated suddenly beneath my breath.

“What is, _mon ami_?” asked Poirot placidly, from the depths of
his comfortable chair.

“Deduce, Poirot, from the following facts! Here is a young lady,
richly dressed—fashionable hat, magnificent furs. She is coming
along slowly, looking up at the houses as she goes. Unknown to her,
she is being shadowed by thr

In [ ]:
len(text)

296401

In [ ]:
# Remove the word "contents" at the beginning
if text.lower().startswith("contents"):
    text = text[len("contents"):]

text = text.strip()

print(text[:500])
print("Total characters:", len(text))

I The Adventure of “The Western Star”

  II The Tragedy at Marsdon Manor

  III The Adventure of the Cheap Flat

  IV The Mystery of Hunter’s Lodge

  V The Million Dollar Bond Robbery

  VI The Adventure of the Egyptian Tomb

  VII Jewel Robbery at the _Grand Metropolitan_

  VIII The Kidnapped Prime Minister

  IX The Disappearance of Mr. Davenheim

  X The Adventure of the Italian Nobleman

  XI The Case of the Missing Will




  POIROT INVESTIGATES




  POIROT INVESTIGATES


  I


  The Adv
Total characters: 296383


In [ ]:
# Lowercase
text = text.lower()

# Keep letters, numbers, punctuation
text = re.sub(r"[^a-zA-Z0-9\s\.\,\!\?\;\:\'\"]", " ", text)

# Remove extra spaces
text = re.sub(r"\s+", " ", text)

text = text.strip()

print(text[:500])

i the adventure of the western star ii the tragedy at marsdon manor iii the adventure of the cheap flat iv the mystery of hunter s lodge v the million dollar bond robbery vi the adventure of the egyptian tomb vii jewel robbery at the grand metropolitan viii the kidnapped prime minister ix the disappearance of mr. davenheim x the adventure of the italian nobleman xi the case of the missing will poirot investigates poirot investigates i the adventure of the western star i was standing at the windo


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts([text])

word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1

print("Final Vocabulary Size:", VOCAB_SIZE)

Final Vocabulary Size: 5803


In [ ]:
sequences = tokenizer.texts_to_sequences([text])[0]

print("Total word tokens:", len(sequences))
print("First 20 tokens:", sequences[:20])

Total word tokens: 53930
First 20 tokens: [3, 2, 751, 6, 2, 842, 370, 843, 2, 1130, 23, 844, 456, 2085, 2, 751, 6, 2, 953, 178]


In [ ]:
import numpy as np

SEQ_LENGTH = 25

inputs = []
targets = []

for i in range(SEQ_LENGTH, len(sequences)):
    inputs.append(sequences[i-SEQ_LENGTH:i])
    targets.append(sequences[i])

inputs = np.array(inputs)
targets = np.array(targets)

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)

Input shape: (53905, 25)
Target shape: (53905,)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

EMBED_DIM = 256
LSTM_UNITS = 256

model = tf.keras.Sequential([

    layers.Input(shape=(SEQ_LENGTH,)),

    # Convert word IDs into dense vectors
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM),
    # First LSTM layer
    layers.LSTM(LSTM_UNITS, return_sequences=True),
    layers.Dropout(0.3),

    # Second LSTM layer
    layers.LSTM(LSTM_UNITS),
    layers.Dropout(0.3),

    # Output layer
    layers.Dense(VOCAB_SIZE, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 25, 256)        │     1,485,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 25, 256)        │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 25, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5803)           │     1,491,371 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,027,563 (15.36 MB)

 Trainable params: 4,027,563 (15.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    inputs,
    targets,
    test_size=0.1,
    random_state=42
)

print("Train shape:", x_train.shape)
print("Validation shape:", x_val.shape)

Train shape: (48501, 40)
Validation shape: (5389, 40)


In [ ]:
BATCH_SIZE = 128

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(10000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

In [ ]:
checkpoint_path = "/content/drive/MyDrive/NLP_dataset/checkpoints/lstm_model.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        save_best_only=True,
        monitor="loss",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="loss",
        patience=3,
        restore_best_weights=True
    )
]

In [ ]:
EPOCHS = 20

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
375/379 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0545 - loss: 7.0129
Epoch 1: loss improved from inf to 6.70723, saving model to /content/drive/MyDrive/NLP_dataset/checkpoints/lstm_model.keras
379/379 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.0545 - loss: 7.0089 - val_accuracy: 0.0611 - val_loss: 6.5141
Epoch 2/20
379/379 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0625 - loss: 6.3526
Epoch 2: loss improved from 6.70723 to 6.32524, saving model to /content/drive/MyDrive/NLP_dataset/checkpoints/lstm_model.keras
379/379 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.0625 - loss: 6.3525 - val_accuracy: 0.0659 - val_loss: 6.4275
Epoch 3/20
379/379 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.0679 - loss: 6.1772
Epoch 3: loss improved from 6.32524 to 6.14683, saving model to /content/drive/MyDrive/NLP_dataset/checkpoints/lstm_model.keras
379/379 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.0679 - loss: 6.1771 - val_accuracy: 0.0694 - val_loss: 6.3741
Epoch 4/20

In [ ]:
index_to_word = {index: word for word, index in tokenizer.word_index.items()}

In [ ]:
index_to_word[tokenizer.word_index["<OOV>"]] = "<OOV>"

In [ ]:
def sample_top_k(predictions, k=50, temperature=1.0):
    predictions = np.asarray(predictions).astype("float64")

    predictions = np.log(predictions + 1e-8) / temperature
    exp_preds = np.exp(predictions)
    predictions = exp_preds / np.sum(exp_preds)

    # Get top-k indices
    top_k_indices = np.argsort(predictions)[-k:]
    top_k_probs = predictions[top_k_indices]
    top_k_probs = top_k_probs / np.sum(top_k_probs)

    return np.random.choice(top_k_indices, p=top_k_probs)

In [ ]:
def generate_text(model, seed_text, num_words=120, temperature=0.85, k=30):

    generated = seed_text.lower().split()

    for _ in range(num_words):

        token_list = tokenizer.texts_to_sequences([" ".join(generated)])[0]
        token_list = token_list[-SEQ_LENGTH:]

        padded = tf.keras.preprocessing.sequence.pad_sequences(
            [token_list], maxlen=SEQ_LENGTH
        )

        predictions = model.predict(padded, verbose=0)[0]

        # Penalize recently used words
        for recent_word in generated[-5:]:
            recent_index = tokenizer.word_index.get(recent_word)
            if recent_index is not None and recent_index < len(predictions):
                predictions[recent_index] *= 0.5

        next_index = sample_top_k(predictions, k=k, temperature=temperature)
        next_word = index_to_word.get(next_index, "")

        generated.append(next_word)

    return " ".join(generated)

In [ ]:
print(generate_text(model, "poirot stood by the window", num_words=120, temperature=0.85,k=30))

poirot stood by the window that it was true he was arrested as i am afraid what he would know so or two minutes cursing all a small official farm as the cause of the bank with a cavity explained that is it is a pretty gold diamond in the murder and rang us was a minute that we had no deaths in the air which sits in dangling norman and the man came rapidly into the face of method as to the other case he has been stolen that you were not to feel the flat i am left that a quarter which he was at the same of harper had to find them then how you are a man and well as is


In [ ]:
model.save("/content/drive/MyDrive/NLP_dataset/final_lstm_model.keras")

In [ ]:
import pickle

with open("/content/drive/MyDrive/NLP_dataset/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [ ]:
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model("/content/drive/MyDrive/NLP_dataset/final_lstm_model.keras")

while True:
    prompt = input("Enter a prompt (or 'exit'): ")

    if prompt.lower() == "exit":
        break

    output = generate_text(model, prompt, num_words=120, temperature=0.85, k=30)

    print("\nGenerated Text:\n")
    print(output)
    print("\n" + "-"*80 + "\n")

Enter a prompt (or 'exit'): Poirot adjusted his moustache and said

Generated Text:

poirot adjusted his moustache and said him there must have been a key of which he was a very nice one of our own country cells poirot was about endeavouring and unequivocally t his feet he reads a ridiculous gesture in an intensity of warnings you i understand a little detective oui the head i will not conceal me but we was in the room from hunter s dale my illuminating answers in a case is a good cat but we will say it because i am sorry to be one of the bank to keep it up the diamond and at a small black hall which ran to the occasion the habit of excellent single in bitter coloured watch and then he were not tactful

--------------------------------------------------------------------------------

Enter a prompt (or 'exit'): exit


# Report

A word-level language model was developed using the novel “Poirot Investigates” by Agatha Christie. The text was first cleaned by removing Project Gutenberg metadata and normalizing whitespace and punctuation. Word-level tokenization was applied using Keras’ Tokenizer, producing a vocabulary of 5,803 unique words and 53,930 total tokens.

Training samples were generated using a batches window of 25 words, where the model learns to predict the next word given the previous context. The architecture consists of an embedding layer (256 dimensions), followed by two stacked LSTM layers (256 units each) with dropout regularization (0.3) and a final dense softmax output layer. The total parameter count is approximately 4 million.

The model was trained using the Adam optimizer with a learning rate of 0.001 and sparse categorical cross-entropy loss. A 90/10 train-validation split was used to monitor generalization. Top-k sampling and temperature scaling were implemented during runs to improve generation quality and reduce repetition.

While the model captures stylistic elements of Agatha Christie’s writing, including recurring characters and dialogue structure, it demonstrates limited long-range consistency compared to large transformer-based models such as ChatGPT or Copilot. This is expected due to the significantly smaller dataset and parameter count.

To compare performance, both models were prompted with:-
***"Poirot adjusted his moustache and said"***

Output – LSTM Model :

poirot adjusted his moustache and said him there must have been a key of which he was a very nice one of our own country cells poirot was about endeavouring and unequivocally t his feet he reads a ridiculous gesture in an intensity of warnings you i understand a little detective oui the head i will not conceal me but we was in the room from hunter s dale my illuminating answers in a case is a good cat but we will say it because i am sorry to be one of the bank to keep it up the diamond and at a small black hall which ran to the occasion the habit of excellent single in bitter coloured watch and then he were not tactful

Output – Copilot (Large Language Model):

Poirot adjusted his moustache and said, “The murderer is the one who believed a small lie would stay small.” He turned sharply toward the fireplace, his gaze settling on the Colonel. “You stopped the clock to frame another, yet you forgot the dust that betrayed you.” The Colonel’s face drained of colour as Poirot continued, “Your alibi collapses, your motive stands alone, and your silence now speaks louder than any denial.” The room held its breath as the truth settled like a final, immovable weight.

The LLm (Copilot) produces grammatically correct, semantically coherent and narratively structured text. It introduces a clear conflict, dialogue structure and logical reasoning within a short passage. The generated paragraph resembles a complete scene with a beginning, development and conclusion.

In contrast, the LSTM-based coursework model demonstrates recognition of stylistic vocabulary such as references to diamonds, rooms and investigative language. However, grammatical consistency worsen over longer sequences and the text lacks strong semantic continuity. This is due to the model learning statistical word transitions from a single novel (53,930 tokens) rather than deep semantic reasoning.

The differences mostly arise from scale and architecture. our model contains approximately 4 million parameters and uses sequential LSTM layers, which have limited long-range contextual memory. Large language models such as Copilot use transformer architectures with self-attention mechanisms and are trained on massive multi-domain datasets containing billions of words. This enables superior global coherence, logical structure and narrative planning.

Despite these limitations, the LSTM model successfully captures some aspects of Agatha Christie’s vocabulary and tone, demonstrating effective word-level language modelling under constrained data conditions.